# RemixFormer++ | PAD-UFES-20 Clinical Image + Metadata



In [1]:
#!/usr/bin/env python3
"""
PAD-UFES-20 + RemixFormer++ clinical-image + metadata training script.

Purpose
-------
This script is for the standard PAD-UFES-20 layout:

    PAD-UFES-20/
        metadata.csv
        images/
            imgs_part_1/
            imgs_part_2/
            imgs_part_3/

It uses the CLINICAL image branch + METADATA branch of RemixFormer++.
It does NOT pretend that PAD-UFES-20 has dermoscopy images.

Project/test settings
---------------------
* Default epochs = 2 for code testing, as requested.
* Change --epochs 100 for the requested final 100-epoch run.
* No k-fold CV. A single stratified 80/20 train/validation holdout is used.
* Default batch size = 2 for GPU memory safety.
* AMP is disabled by default to avoid PyTorch checkpoint/autograd recomputation
  mismatches seen in some Colab/Kaggle PyTorch builds. Use --amp only if needed.

Architecture implemented here
-----------------------------
Clinical image:
    896x896 clinical image
        -> 384x384 global view
        -> Swin-T global feature map
        -> attention-based lesion proposal / LSM
        -> four 224x224 high-resolution lesion crops
        -> Swin-T local encoder
        -> cross-scale global/local attention fusion (CSF)
        -> clinical representation gC

Metadata:
    structured metadata
        -> one-hot encoding
        -> Linear -> BatchNorm -> ReLU
        -> metadata representation gM

Final PAD model:
    [gC, gM]
        -> cross-modality attention fusion
        -> classifier

Paper-faithfulness notes
------------------------
The paper describes a custom Swin block with a CLS token for the clinical
branch and a differentiable attention-sampling LSM. Reproducing the custom
attention internals exactly would require reimplementing the paper's modified
Swin WMSA. This implementation preserves the published clinical pipeline and
feature dimensions, but uses a practical attention-score proposal mechanism
on the final Swin feature map and cross-attention CSF/CMF modules.

The paper reports 200 epochs, batch size 64 on 4 V100 GPUs, SGD, cosine LR,
initial LR 1e-4, ImageNet initialization, and end-to-end cross-entropy. This
project intentionally uses 2 epochs by default for testing and no CV.
"""

from __future__ import annotations

import json
import math
import os
import random
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import swin_t, Swin_T_Weights

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

@dataclass
class Config:
    data_root: str = "/kaggle/input/datasets/abtahimajeed/pad-ufes-20"
    metadata_csv: str = ""  # auto-detect metadata.csv
    output_dir: str = "/kaggle/working/remix_padufes20_clinical_metadata"

    epochs: int = 50
    batch_size: int = 2
    num_workers: int = 2
    lr: float = 1e-4
    weight_decay: float = 0.0
    seed: int = 42

    image_size: int = 896
    global_size: int = 384
    local_size: int = 224
    local_top_k: int = 4

    swin_pretrained: bool = True
    gradient_checkpointing: bool = False

    amp: bool = False
    grad_accum: int = 1
    max_grad_norm: float = 1.0

    val_fraction: float = 0.20
    max_samples: int = 0
    save_every: int = 1

    feature_dim: int = 768
    metadata_dim: int = 768
    fusion_heads: int = 8




In [2]:
PAPER = {
    "dataset": "PAD-UFES-20",
    "PAD_UFES20_samples": 2298,
    "PAD_UFES20_diagnoses": ["BCC", "SCC", "AK", "SK", "Mel", "Nev"],
    "clinical_input_reference": "896x896 then 384x384 global input",
    "local_patch": "224x224",
    "LSM_top_k": 4,
    "clinical_backbone": "Swin-T",
    "metadata_encoding": "one-hot",
    "metadata_branch": "Linear -> BatchNorm -> ReLU",
    "epochs_reference": 200,
    "project_epochs_default": 2,
    "project_final_epochs": 100,
    "cross_validation_in_project": False,
    "reference_batch_size": 64,
    "optimizer": "SGD",
    "initial_lr": 1e-4,
    "scheduler": "cosine",
    "loss": "cross-entropy",
}

In [3]:
# ---------------------------------------------------------------------
# Reproducibility / device

# ---------------------------------------------------------------------

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# ---------------------------------------------------------------------
# Dataset discovery

# ---------------------------------------------------------------------

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

IMAGE_COLUMN_CANDIDATES = [
    "img_id", "image_id", "image", "image_name", "filename", "file_name",
    "path", "image_path", "clinical_image", "clinical_image_path",
]

LABEL_COLUMN_CANDIDATES = [
    "diagnostic", "diagnosis", "diagnostic_label", "label", "target",
    "class", "lesion_type", "disease", "dx",
]

ID_LIKE = {
    "patient_id", "lesion_id", "image_id", "img_id", "case_id", "subject_id",
    "file_name", "filename", "image_name", "path", "image_path",
}


def normalize_col(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")


def list_images(root: Path) -> List[Path]:
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )


def build_image_index(root: Path) -> Dict[str, Path]:
    index: Dict[str, Path] = {}
    for p in list_images(root):
        index[p.name.lower()] = p
        index[p.stem.lower()] = p
    return index


def choose_column(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    normalized = {normalize_col(c): c for c in df.columns}
    for c in candidates:
        key = normalize_col(c)
        if key in normalized:
            return normalized[key]
    return None


def find_metadata_csv(root: Path, explicit: str = "") -> Path:
    if explicit:
        p = Path(explicit)
        if not p.is_absolute():
            p = root / p
        if not p.exists():
            raise FileNotFoundError(f"Metadata CSV not found: {p}")
        return p

    preferred = root / "metadata.csv"
    if preferred.exists():
        return preferred

    csvs = sorted(root.rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV file found under {root}")

    for p in csvs:
        if p.name.lower() == "metadata.csv":
            return p

    # Fall back to the CSV containing both an image-like and label-like column.
    scored = []
    for p in csvs:
        try:
            df = pd.read_csv(p, low_memory=False)
        except Exception:
            continue
        score = 0
        if choose_column(df, IMAGE_COLUMN_CANDIDATES):
            score += 5
        if choose_column(df, LABEL_COLUMN_CANDIDATES):
            score += 5
        scored.append((score, p))
    if not scored:
        raise RuntimeError("CSV files exist but none could be read.")
    scored.sort(key=lambda x: (x[0], x[1].name), reverse=True)
    return scored[0][1]


def infer_label_column(df: pd.DataFrame) -> str:
    col = choose_column(df, LABEL_COLUMN_CANDIDATES)
    if col:
        return col

    candidates = []
    for c in df.columns:
        nc = normalize_col(c)
        if nc in ID_LIKE or nc.endswith("_id"):
            continue
        nunique = df[c].nunique(dropna=True)
        if 2 <= nunique <= min(30, max(2, len(df) // 3)):
            candidates.append((nunique, c))
    if candidates:
        candidates.sort()
        return candidates[0][1]

    raise RuntimeError(
        "Could not infer diagnostic label column. "
        "Pass --label-column explicitly."
    )


def resolve_image(value: object, csv_dir: Path, root: Path, index: Dict[str, Path]) -> Optional[Path]:
    if pd.isna(value):
        return None
    s = str(value).strip()
    if not s:
        return None

    candidates = [
        Path(s),
        csv_dir / s,
        root / s,
        root / "images" / s,
        root / "images" / "imgs_part_1" / s,
        root / "images" / "imgs_part_2" / s,
        root / "images" / "imgs_part_3" / s,
    ]
    for p in candidates:
        if p.exists() and p.is_file():
            return p.resolve()

    name = Path(s).name.lower()
    if name in index:
        return index[name]
    stem = Path(s).stem.lower()
    if stem in index:
        return index[stem]
    return None


def prepare_dataframe(
    root: Path,
    metadata_csv: str = "",
    explicit_image_column: Optional[str] = None,
    explicit_label_column: Optional[str] = None,
    max_samples: int = 0,
) -> Tuple[pd.DataFrame, Path, str, str]:
    csv_path = find_metadata_csv(root, metadata_csv)
    df = pd.read_csv(csv_path, low_memory=False)
    image_index = build_image_index(root)

    image_col = explicit_image_column or choose_column(df, IMAGE_COLUMN_CANDIDATES)
    label_col = explicit_label_column or infer_label_column(df)

    if image_col is None:
        raise RuntimeError(
            "Could not find the image identifier column in metadata.csv. "
            "Expected one of: " + ", ".join(IMAGE_COLUMN_CANDIDATES) +
            ". Pass --image-column explicitly."
        )

    rows = []
    missing = 0
    for _, r in df.iterrows():
        p = resolve_image(r[image_col], csv_path.parent, root, image_index)
        if p is None:
            missing += 1
            continue
        y = r[label_col]
        if pd.isna(y):
            continue
        item = r.to_dict()
        item["_resolved_image"] = str(p)
        rows.append(item)

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError(
            f"No usable image rows found. CSV={csv_path}, image_column={image_col}."
        )

    out["_label_str"] = out[label_col].astype(str).str.strip()
    out = out[~out["_label_str"].isin({"", "nan", "None"})].reset_index(drop=True)

    if max_samples > 0 and len(out) > max_samples:
        out = out.sample(max_samples, random_state=42).reset_index(drop=True)

    print(f"[DATA] metadata CSV       : {csv_path}")
    print(f"[DATA] CSV rows            : {len(df)}")
    print(f"[DATA] Usable rows         : {len(out)}")
    print(f"[DATA] Missing images      : {missing}")
    print(f"[DATA] Image column        : {image_col}")
    print(f"[DATA] Label column        : {label_col}")
    print("[DATA] Class distribution:")
    print(out["_label_str"].value_counts())
    print(f"[DATA] Image files indexed : {len(image_index)}")

    return out, csv_path, image_col, label_col

In [5]:
# ---------------------------------------------------------------------
# Metadata encoder

# ---------------------------------------------------------------------

class MetadataEncoder:
    """One-hot encoder for structured PAD-UFES-20 metadata.

    The paper specifies one-hot representation for structured metadata.
    Identifier/image/label columns are excluded. Unknown validation values
    are encoded as all-zero for that category.
    """

    def __init__(self, excluded: Sequence[str]):
        self.excluded = {normalize_col(x) for x in excluded}
        self.columns: List[str] = []
        self.categories: Dict[str, List[str]] = {}
        self.output_dim = 0

    def fit(self, df: pd.DataFrame) -> None:
        self.columns = []
        self.categories = {}
        for c in df.columns:
            nc = normalize_col(c)
            if nc.startswith("_") or nc in self.excluded or nc.endswith("_id"):
                continue
            if c in {"_resolved_image", "_label_str"}:
                continue

            s = df[c]
            vals = s.astype(str).replace({"nan": "__MISSING__", "None": "__MISSING__"})
            nunique = vals.nunique(dropna=False)
            if 1 <= nunique <= 128:
                cats = sorted(vals.unique().tolist())
                self.columns.append(c)
                self.categories[c] = cats

        self.output_dim = sum(len(self.categories[c]) for c in self.columns)
        if self.output_dim == 0:
            warnings.warn(
                "No usable metadata columns were found. The metadata branch will "
                "receive an empty vector."
            )

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        parts: List[np.ndarray] = []
        for c in self.columns:
            cats = self.categories[c]
            lookup = {v: i for i, v in enumerate(cats)}
            vals = df[c].astype(str).replace({"nan": "__MISSING__", "None": "__MISSING__"}).tolist()
            arr = np.zeros((len(df), len(cats)), dtype=np.float32)
            for i, v in enumerate(vals):
                j = lookup.get(v)
                if j is not None:
                    arr[i, j] = 1.0
            parts.append(arr)
        if not parts:
            return np.zeros((len(df), 0), dtype=np.float32)
        return np.concatenate(parts, axis=1)

    def save(self, path: Path) -> None:
        payload = {
            "columns": self.columns,
            "categories": self.categories,
            "output_dim": self.output_dim,
        }
        path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

In [6]:
# ---------------------------------------------------------------------
# Augmentation

# ---------------------------------------------------------------------

class RandomRotationMultiplesOfTen:
    def __init__(self, max_degrees: int = 180):
        self.angles = list(range(0, max_degrees + 1, 10))

    def __call__(self, img):
        return transforms.functional.rotate(
            img,
            random.choice(self.angles),
            interpolation=InterpolationMode.BILINEAR,
            fill=0,
        )


def build_train_transform(size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomResizedCrop(
            size,
            scale=(0.65, 1.0),
            ratio=(1.0, 1.0),
            interpolation=InterpolationMode.BICUBIC,
        ),
        RandomRotationMultiplesOfTen(180),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.15, 0.15),
            shear=(-10.0, 10.0),
            interpolation=InterpolationMode.BILINEAR,
            fill=0,
        ),
        transforms.ColorJitter(
            brightness=(0.5, 1.5),
            contrast=(0.5, 1.5),
            saturation=(0.5, 1.5),
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])


def build_eval_transform(size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((size, size), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])

In [7]:
# ---------------------------------------------------------------------
# Dataset

# ---------------------------------------------------------------------

class PADUFES20Dataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        label_to_idx: Dict[str, int],
        transform,
        metadata: np.ndarray,
    ):
        self.frame = frame.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform
        self.metadata = metadata

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        path = row["_resolved_image"]
        try:
            image = Image.open(path).convert("RGB")
        except Exception as exc:
            raise RuntimeError(f"Could not read image: {path}") from exc

        image = self.transform(image)
        label = self.label_to_idx[row["_label_str"]]
        meta = torch.from_numpy(self.metadata[idx]).float()

        return {
            "image": image,
            "metadata": meta,
            "label": torch.tensor(label, dtype=torch.long),
            "path": path,
        }

In [8]:
# ---------------------------------------------------------------------
# Clinical branch with explicit ImageNet initialization + attention capture
# ---------------------------------------------------------------------

class SwinFeatureBackbone(nn.Module):
    """Swin-T feature extractor returning the final spatial feature map."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
        self.swin = swin_t(weights=weights)
        self.dim = 768
        self.imagenet_pretrained = bool(pretrained)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.swin.features(x)
        x = self.swin.norm(x)
        return x


class LesionSelectionModule(nn.Module):
    """LSM-style lesion proposal module with saved attention scores."""
    def __init__(self, dim: int = 768, top_k: int = 4):
        super().__init__()
        self.top_k = top_k
        self.score = nn.Linear(dim, 1)

    def forward(self, feature_map, original_images, global_size, original_size, local_size):
        B, H, W, D = feature_map.shape
        raw_scores = self.score(feature_map).squeeze(-1)
        pooled = F.avg_pool2d(raw_scores.unsqueeze(1), kernel_size=3, stride=1).squeeze(1)
        PH, PW = pooled.shape[-2:]
        flat = pooled.flatten(1)
        k = min(self.top_k, flat.shape[1])
        top_scores, top_idx = torch.topk(flat, k=k, dim=1)

        cell = float(original_size) / float(H)
        crop_size = local_size
        crops, boxes = [], []
        for b in range(B):
            for j in range(k):
                idx = int(top_idx[b, j].item())
                rr, cc = idx // PW, idx % PW
                center_r = (rr + 1.5) * cell
                center_c = (cc + 1.5) * cell
                y1 = int(round(center_r - crop_size / 2))
                x1 = int(round(center_c - crop_size / 2))
                y1 = max(0, min(original_size - crop_size, y1))
                x1 = max(0, min(original_size - crop_size, x1))
                y2, x2 = y1 + crop_size, x1 + crop_size
                crop = original_images[b:b+1, :, y1:y2, x1:x2]
                if crop.shape[-2:] != (local_size, local_size):
                    crop = F.interpolate(crop, size=(local_size, local_size), mode='bilinear', align_corners=False)
                crops.append(crop)
                boxes.append([b, x1, y1, x2, y2])

        return {
            'crops': torch.cat(crops, dim=0),
            'scores': top_scores,
            'indices': top_idx,
            'boxes': torch.tensor(boxes, device=feature_map.device, dtype=torch.float32),
            'attention_map': raw_scores,
            'pooled_attention_map': pooled,
        }


class CrossScaleFusion(nn.Module):
    """Cross-attention between global clinical tokens and local lesion tokens."""
    def __init__(self, dim: int = 768, heads: int = 8):
        super().__init__()
        self.global_norm = nn.LayerNorm(dim)
        self.local_norm = nn.LayerNorm(dim)
        self.global_to_local = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.local_to_global = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.g_out = nn.Linear(dim, dim)
        self.l_out = nn.Linear(dim, dim)

    def forward(self, global_tokens, local_tokens, return_attention=False):
        gq = self.global_norm(global_tokens)
        lk = self.local_norm(local_tokens)
        g_cross, g_attn = self.global_to_local(gq, lk, lk, need_weights=return_attention)
        l_cross, l_attn = self.local_to_global(lk, gq, gq, need_weights=return_attention)
        g = global_tokens + self.g_out(g_cross)
        l = local_tokens + self.l_out(l_cross)
        if return_attention:
            return g.mean(dim=1), l.mean(dim=1), g_attn, l_attn
        return g.mean(dim=1), l.mean(dim=1)


class ClinicalBranch(nn.Module):
    def __init__(self, pretrained=True, top_k=4, fusion_heads=8):
        super().__init__()
        self.global_backbone = SwinFeatureBackbone(pretrained=pretrained)
        self.local_backbone = SwinFeatureBackbone(pretrained=pretrained)
        self.lsm = LesionSelectionModule(dim=768, top_k=top_k)
        self.csf = CrossScaleFusion(dim=768, heads=fusion_heads)

    def forward(self, images_896, global_size, local_size, return_attention=False):
        global_images = F.interpolate(images_896, size=(global_size, global_size), mode='bilinear', align_corners=False)
        gmap = self.global_backbone(global_images)
        B, H, W, D = gmap.shape
        global_tokens = gmap.reshape(B, H * W, D)
        proposals = self.lsm(gmap, images_896, global_size, images_896.shape[-1], local_size)

        local_map = self.local_backbone(proposals['crops'])
        local_tokens_all = local_map.reshape(local_map.shape[0], -1, D).mean(dim=1)
        K = proposals['crops'].shape[0] // B
        local_tokens = local_tokens_all.reshape(B, K, D)

        if return_attention:
            gC, lC, g_attn, l_attn = self.csf(global_tokens, local_tokens, return_attention=True)
        else:
            gC, lC = self.csf(global_tokens, local_tokens, return_attention=False)
            g_attn = l_attn = None

        return {
            'gC': gC, 'lC': lC, 'global_feature_map': gmap, 'global_tokens': global_tokens,
            'local_tokens': local_tokens, 'lesion_crops': proposals['crops'],
            'lesion_boxes': proposals['boxes'], 'lesion_scores': proposals['scores'],
            'lsm_attention_map': proposals['attention_map'],
            'lsm_pooled_attention_map': proposals['pooled_attention_map'],
            'csf_global_to_local_attention': g_attn, 'csf_local_to_global_attention': l_attn,
        }


In [9]:
# =========================
# KAGGLE NOTEBOOK SETTINGS
# =========================

DATA_ROOT = "/kaggle/input/datasets/abtahimajeed/pad-ufes-20"
OUTPUT_DIR = "/kaggle/working/remix_padufes20_clinical_metadata_50ep_final"

EPOCHS = 50            
BATCH_SIZE = 2
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
SEED = 42
MAX_SAMPLES = 0         
LOCAL_TOP_K = 4
USE_PRETRAINED = True  # REQUIRED: ImageNet-pretrained Swin-T
USE_AMP = False        
GRAD_ACCUM = 1

cfg = Config(
    data_root=DATA_ROOT,
    metadata_csv="",
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
    swin_pretrained=USE_PRETRAINED,
    amp=USE_AMP,
    grad_accum=GRAD_ACCUM,
    max_samples=MAX_SAMPLES,
    local_top_k=LOCAL_TOP_K,
)

seed_everything(cfg.seed)
device = get_device()
root = Path(cfg.data_root)
output = Path(cfg.output_dir)
output.mkdir(parents=True, exist_ok=True)

print("=" * 95)
print("RemixFormer++ | PAD-UFES-20 Clinical Image + Metadata")
print("=" * 95)
print(f"Device                         : {device}")
if device.type == "cuda":
    print(f"GPU                            : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version                   : {torch.version.cuda}")
print(f"Dataset root                  : {root}")
print(f"Epochs                        : {cfg.epochs}")
if cfg.epochs != 50:
    raise RuntimeError(f"This is the real 50-epoch notebook, but EPOCHS={cfg.epochs}. Stop and fix the configuration before training.")
print(f"Batch size                    : {cfg.batch_size}")
print(f"Learning rate                 : {cfg.lr}")
print(f"CUDA AMP                      : {cfg.amp}")
print(f"Output directory              : {output}")
print("=" * 95)

if not root.exists():
    raise FileNotFoundError(
        f"Dataset root does not exist: {root}\n"
        "Check the Kaggle dataset attachment and update DATA_ROOT above."
    )


RemixFormer++ | PAD-UFES-20 Clinical Image + Metadata
Device                         : cpu
Dataset root                  : \kaggle\input\datasets\abtahimajeed\pad-ufes-20
Epochs                        : 50
Batch size                    : 2
Learning rate                 : 0.0001
CUDA AMP                      : False
Output directory              : \kaggle\working\remix_padufes20_clinical_metadata_50ep_final


FileNotFoundError: Dataset root does not exist: \kaggle\input\datasets\abtahimajeed\pad-ufes-20
Check the Kaggle dataset attachment and update DATA_ROOT above.

In [ ]:
# ---------------------------------------------------------------------
# ImageNet initialization verification
# ---------------------------------------------------------------------

print("\nIMAGENET INITIALIZATION")
print("-" * 80)
print("Clinical global backbone : torchvision Swin-T")
print("Clinical local backbone  : torchvision Swin-T")
print("Weights                  : Swin_T_Weights.IMAGENET1K_V1")
print("ImageNet pretrained      :", cfg.swin_pretrained)
print("Feature dimension        : 768")
print("Global input             : 384 x 384")
print("Local input              : 224 x 224")
print("-" * 80)

if cfg.swin_pretrained:
    print("ImageNet pretrained weights will be loaded for BOTH clinical Swin-T backbones.")
else:
    print("WARNING: ImageNet initialization is disabled.")



IMAGENET INITIALIZATION
--------------------------------------------------------------------------------
Clinical global backbone : torchvision Swin-T
Clinical local backbone  : torchvision Swin-T
Weights                  : Swin_T_Weights.IMAGENET1K_V1
ImageNet pretrained      : True
Feature dimension        : 768
Global input             : 384 x 384
Local input              : 224 x 224
--------------------------------------------------------------------------------
ImageNet pretrained weights will be loaded for BOTH clinical Swin-T backbones.


In [ ]:
# ---------------------------------------------------------------------
# Metadata branch + CMF attention capture
# ---------------------------------------------------------------------

class MetadataBranch(nn.Module):
    def __init__(self, input_dim: int, out_dim: int = 768):
        super().__init__()
        self.input_dim, self.out_dim = input_dim, out_dim
        if input_dim == 0:
            self.net = None
        else:
            self.net = nn.Sequential(nn.Linear(input_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(inplace=True))

    def forward(self, x):
        if self.net is None:
            return torch.zeros(x.shape[0], self.out_dim, device=x.device, dtype=x.dtype)
        return self.net(x)


class CrossModalityFusion(nn.Module):
    def __init__(self, dim=768, heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))

    def forward(self, clinical, metadata, return_attention=False):
        tokens = torch.stack([clinical, metadata], dim=1)
        q = self.norm(tokens)
        fused, attn = self.attn(q, q, q, need_weights=return_attention)
        tokens = tokens + fused
        tokens = tokens + self.ffn(tokens)
        result = tokens.mean(dim=1)
        return (result, attn) if return_attention else result


class RemixPADClinicalMetadata(nn.Module):
    def __init__(self, num_classes, metadata_input_dim, cfg):
        super().__init__()
        self.clinical = ClinicalBranch(pretrained=cfg.swin_pretrained, top_k=cfg.local_top_k, fusion_heads=cfg.fusion_heads)
        self.metadata = MetadataBranch(metadata_input_dim, out_dim=cfg.metadata_dim)
        self.cmf = CrossModalityFusion(dim=cfg.feature_dim, heads=cfg.fusion_heads)
        self.classifier = nn.Linear(cfg.feature_dim, num_classes)

    def forward(self, images, metadata, cfg, return_attention=False):
        clinical_out = self.clinical(images, cfg.global_size, cfg.local_size, return_attention=return_attention)
        gM = self.metadata(metadata)
        if return_attention:
            fused, cmf_attn = self.cmf(clinical_out['gC'], gM, return_attention=True)
        else:
            fused = self.cmf(clinical_out['gC'], gM, return_attention=False)
            cmf_attn = None
        logits = self.classifier(fused)
        clinical_out.update({'gM': gM, 'fused': fused, 'logits': logits, 'cmf_attention': cmf_attn})
        return clinical_out


In [ ]:
# ---------------------------------------------------------------------
# Metrics / visualization
# ---------------------------------------------------------------------

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def save_confusion_matrix(y_true, y_pred, class_names, path: Path, title: str):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm)
    fig.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(class_names)), yticks=np.arange(len(class_names)),
        xticklabels=class_names, yticklabels=class_names,
        xlabel="Predicted", ylabel="True", title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    threshold = cm.max() / 2.0 if cm.size else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, str(int(cm[i, j])), ha="center", va="center",
                fontsize=12, fontweight="bold",
                color="white" if cm[i, j] > threshold else "black",
            )
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return cm

def print_detailed_results(y_true, y_pred, class_names, prefix="VALIDATION"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    report = classification_report(
        y_true, y_pred, labels=list(range(len(class_names))), target_names=class_names,
        digits=4, zero_division=0
    )
    print(f"\n{'=' * 95}\n{prefix} DETAILED RESULTS\n{'=' * 95}")
    print(report)
    print("Confusion Matrix (rows=True, columns=Predicted):")
    print("Rows = True class | Columns = Predicted class")
    print(cm)
    print("Numeric cell counts are also written inside the saved confusion-matrix figure.")
    return cm

def save_training_preview(dataset: Dataset, path: Path, n: int = 8):
    n = min(n, len(dataset))
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = np.asarray(axes).reshape(-1)
    for i, ax in enumerate(axes):
        if i < n:
            item = dataset[i]
            img = denormalize(item["image"]).permute(1, 2, 0).cpu().numpy()
            ax.imshow(img)
            ax.set_title(f"label={item['label'].item()}")
        ax.axis("off")
    fig.suptitle("PAD-UFES-20 training augmentation preview")
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

def save_lsm_preview(image: torch.Tensor, boxes: torch.Tensor, path: Path):
    img = denormalize(image).permute(1, 2, 0).cpu().numpy()
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img)
    for i, box in enumerate(boxes.detach().cpu().numpy()):
        _, x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2)
        ax.add_patch(rect)
        ax.text(x1 + 4, y1 + 16, str(i), bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))
    ax.set_title("LSM lesion proposals")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

def denormalize(x: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)


In [ ]:
# ---------------------------------------------------------------------
# Single-image intermediate visualization
# ---------------------------------------------------------------------

def _norm01(a):
    a = np.asarray(a, dtype=np.float32)
    lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo + 1e-8)

def _to_display_image(t):
    return denormalize(t.detach().cpu()).permute(1, 2, 0).numpy().clip(0, 1)

def save_single_image_intermediates(model, dataset, frame, label_to_idx, device, cfg, out_dir, image_index=0, stage='trained'):
    out_dir = Path(out_dir) / 'single_image_intermediate' / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    item = dataset[image_index]
    image = item['image'].unsqueeze(0).to(device)
    metadata = item['metadata'].unsqueeze(0).to(device)
    true_label = int(item['label'].item())

    model.eval()
    with torch.no_grad():
        out = model(image, metadata, cfg, return_attention=True)

    original = _to_display_image(image[0])
    boxes = out['lesion_boxes'][out['lesion_boxes'][:, 0] == 0].detach().cpu().numpy()
    scores = out['lesion_scores'][0].detach().cpu().numpy()
    att = out['lsm_attention_map'][0].detach().cpu().numpy()
    pooled = out['lsm_pooled_attention_map'][0].detach().cpu().numpy()
    gmap = out['global_feature_map'][0].detach().cpu().numpy()
    global_activation = np.mean(np.abs(gmap), axis=-1)
    local_tokens = out['local_tokens'][0].detach().cpu().numpy()
    local_activation = np.mean(np.abs(local_tokens), axis=-1)
    logits = out['logits'][0].detach().cpu().numpy()
    probs = torch.softmax(out['logits'][0], dim=0).detach().cpu().numpy()
    class_names = [x for x, _ in sorted(label_to_idx.items(), key=lambda z: z[1])]

    # 1. Original image
    fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(original); ax.axis('off'); ax.set_title(f'Original clinical image | true={class_names[true_label]}')
    fig.tight_layout(); fig.savefig(out_dir / '01_original.png', dpi=200); plt.close(fig)

    # 2. LSM raw attention score as image
    fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(original); ax.imshow(_norm01(att), cmap='jet', alpha=0.48, extent=(0, original.shape[1], original.shape[0], 0), interpolation='bilinear'); ax.axis('off'); ax.set_title('02 LSM attention score heatmap')
    fig.tight_layout(); fig.savefig(out_dir / '02_lsm_attention_score_heatmap.png', dpi=200); plt.close(fig)

    # 3. Attention sampling / selected boxes
    fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(original)
    for j, b in enumerate(boxes):
        _, x1, y1, x2, y2 = b
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, linewidth=3)
        ax.add_patch(rect); ax.text(x1+5, y1+22, f'P{j+1}: {scores[j]:.4f}', bbox=dict(facecolor='white', alpha=0.75, edgecolor='none'))
    ax.axis('off'); ax.set_title('03 LSM attention sampling with 4 lesion boxes')
    fig.tight_layout(); fig.savefig(out_dir / '03_attention_sampling_boxes.png', dpi=200); plt.close(fig)

    # 4. Pooled LSM score map
    fig, ax = plt.subplots(figsize=(8, 8)); im=ax.imshow(_norm01(pooled), cmap='jet'); ax.set_title('04 Pooled 3x3 LSM attention score'); ax.axis('off'); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(out_dir / '04_lsm_pooled_score.png', dpi=200); plt.close(fig)

    # 5. Four selected crops
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    crops = out['lesion_crops'].detach().cpu()
    for j, ax in enumerate(axes):
        if j < crops.shape[0]: ax.imshow(_to_display_image(crops[j])); ax.set_title(f'Crop {j+1} | score={scores[j]:.4f}')
        ax.axis('off')
    fig.tight_layout(); fig.savefig(out_dir / '05_selected_lesion_crops.png', dpi=200); plt.close(fig)

    # 6. Global Swin activation map
    fig, ax = plt.subplots(figsize=(8, 8)); im=ax.imshow(_norm01(global_activation), cmap='viridis'); ax.set_title('06 Global Swin-T feature activation'); ax.axis('off'); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(out_dir / '06_global_feature_activation.png', dpi=200); plt.close(fig)

    # 7. Local proposal feature activations
    fig, ax = plt.subplots(figsize=(9, 4)); ax.bar(np.arange(len(local_activation))+1, local_activation); ax.set_xlabel('Lesion proposal'); ax.set_ylabel('Mean |feature|'); ax.set_title('07 Local lesion feature activation'); ax.set_xticks(np.arange(len(local_activation))+1)
    fig.tight_layout(); fig.savefig(out_dir / '07_local_feature_activation.png', dpi=200); plt.close(fig)

    # 8. CSF attention: global -> local
    g2l = out['csf_global_to_local_attention'][0].detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(8, 6)); im=ax.imshow(g2l, aspect='auto', cmap='viridis'); ax.set_xlabel('Local lesion token'); ax.set_ylabel('Global token'); ax.set_title('08 CSF attention: global → local'); fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(out_dir / '08_csf_global_to_local_attention.png', dpi=200); plt.close(fig)

    # 9. CSF attention: local -> global summarized over global tokens
    l2g = out['csf_local_to_global_attention'][0].detach().cpu().numpy()
    l2g_mean = l2g.mean(axis=-1)
    fig, ax = plt.subplots(figsize=(8, 4)); x=np.arange(len(l2g_mean))+1; ax.bar(x, l2g_mean); ax.set_xlabel('Local lesion token'); ax.set_ylabel('Mean attention'); ax.set_title('09 CSF attention: local → global'); ax.set_xticks(x); [ax.text(x[k], l2g_mean[k]+0.00015, f'{l2g_mean[k]:.4f}', ha='center', va='bottom', fontsize=10) for k in range(len(l2g_mean))]
    fig.tight_layout(); fig.savefig(out_dir / '09_csf_local_to_global_attention.png', dpi=200); plt.close(fig)

    # 10. Clinical ↔ metadata CMF attention
    cmf = out['cmf_attention'][0].detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(7, 6)); im=ax.imshow(cmf, cmap='viridis', vmin=0.0, vmax=1.0); ax.set_xticks([0,1], ['Clinical','Metadata']); ax.set_yticks([0,1], ['Clinical','Metadata']); ax.set_xlabel('Key modality'); ax.set_ylabel('Query modality'); ax.set_title('05 Clinical ↔ metadata CMF attention');
    for i in range(cmf.shape[0]):
        for j in range(cmf.shape[1]):
            ax.text(j, i, f'{cmf[i,j]:.4f}', ha='center', va='center', fontsize=14, fontweight='bold', color='black' if cmf[i,j] >= 0.5 else 'white')
    fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(out_dir / '10_cmf_attention.png', dpi=200, bbox_inches='tight'); plt.close(fig)

    # 11. Metadata embedding
    gm = out['gM'][0].detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(12, 4)); ax.plot(gm); ax.set_xlabel('Feature index'); ax.set_ylabel('Activation'); ax.set_title('11 Metadata representation gM (768-D)'); fig.tight_layout(); fig.savefig(out_dir / '11_metadata_feature_gM.png', dpi=200); plt.close(fig)

    # 12. Fused representation
    fused = out['fused'][0].detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(12, 4)); ax.plot(fused); ax.set_xlabel('Feature index'); ax.set_ylabel('Activation'); ax.set_title('12 Fused clinical + metadata representation (768-D)'); fig.tight_layout(); fig.savefig(out_dir / '12_fused_feature.png', dpi=200); plt.close(fig)

    # 13. Diagnosis probabilities
    fig, ax = plt.subplots(figsize=(10, 5)); ax.bar(class_names, probs); ax.set_ylim(0,1); ax.set_ylabel('Probability'); ax.set_title(f'13 Diagnosis output | predicted={class_names[int(probs.argmax())]}');
    for i,p in enumerate(probs): ax.text(i,p+0.02,f'{p:.3f}',ha='center')
    fig.tight_layout(); fig.savefig(out_dir / '13_diagnosis_probabilities.png', dpi=200); plt.close(fig)

    # 14. Save numeric intermediate values
    np.save(out_dir / 'lsm_attention_map.npy', att)
    np.save(out_dir / 'lsm_pooled_attention_map.npy', pooled)
    np.save(out_dir / 'lsm_selected_scores.npy', scores)
    np.save(out_dir / 'csf_global_to_local_attention.npy', g2l)
    np.save(out_dir / 'csf_local_to_global_attention.npy', l2g)
    np.save(out_dir / 'cmf_attention.npy', cmf)
    (out_dir / 'intermediate_summary.json').write_text(json.dumps({
        'dataset_index': int(image_index), 'image_path': str(item['path']), 'true_class': class_names[true_label],
        'predicted_class': class_names[int(probs.argmax())], 'probabilities': {class_names[i]: float(probs[i]) for i in range(len(class_names))},
        'input_shape': list(image.shape), 'global_feature_map': list(out['global_feature_map'].shape),
        'local_tokens': list(out['local_tokens'].shape), 'gC': list(out['gC'].shape), 'gM': list(out['gM'].shape), 'fused': list(out['fused'].shape),
        'lsm_top_scores': [float(x) for x in scores], 'image_index_fixed_for_visualization': int(image_index)
    }, indent=2), encoding='utf-8')

    print(f'[INTERMEDIATE] Saved single-image visualizations to: {out_dir}')
    print(f'[INTERMEDIATE] Image: {item["path"]}')
    print(f'[INTERMEDIATE] True={class_names[true_label]} | Predicted={class_names[int(probs.argmax())]}')
    print(f'[INTERMEDIATE] LSM selected scores: {scores}')
    print(f'[INTERMEDIATE] LSM raw map shape: {att.shape} | min={att.min():.6f} max={att.max():.6f}')
    print(f'[INTERMEDIATE] LSM pooled map shape: {pooled.shape} | min={pooled.min():.6f} max={pooled.max():.6f}')
    print(f'[INTERMEDIATE] Lesion boxes:\n{boxes}')
    print(f'[INTERMEDIATE] Global feature map shape: {out["global_feature_map"].shape}')
    print(f'[INTERMEDIATE] Global tokens shape: {out["global_tokens"].shape}')
    print(f'[INTERMEDIATE] Local tokens shape: {out["local_tokens"].shape}')
    print(f'[INTERMEDIATE] gC shape: {out["gC"].shape}')
    print(f'[INTERMEDIATE] gM shape: {out["gM"].shape}')
    print(f'[INTERMEDIATE] Fused shape: {out["fused"].shape}')
    print(f'[INTERMEDIATE] CSF global→local attention shape: {out["csf_global_to_local_attention"].shape}')
    print(f'[INTERMEDIATE] CSF local→global attention shape: {out["csf_local_to_global_attention"].shape}')
    print('[INTERMEDIATE] CMF attention matrix (numeric values):')
    for ri, rname in enumerate(['Clinical','Metadata']):
        print(f'    {rname:8s} -> Clinical={cmf[ri,0]:.4f} | Metadata={cmf[ri,1]:.4f}')
    print(f'[INTERMEDIATE] Diagnosis logits: {logits}')
    print(f'[INTERMEDIATE] Diagnosis probabilities: {probs}')
    return out


In [ ]:
# ---------------------------------------------------------------------
# Splitting

# ---------------------------------------------------------------------

def make_split(frame: pd.DataFrame, val_fraction: float, seed: int):
    idx = np.arange(len(frame))
    train_idx, val_idx = train_test_split(
        idx,
        test_size=val_fraction,
        random_state=seed,
        shuffle=True,
        stratify=frame["_label_str"].to_numpy(),
    )
    return train_idx, val_idx


def build_label_map(frame: pd.DataFrame) -> Dict[str, int]:
    labels = sorted(frame["_label_str"].unique().tolist())
    return {name: i for i, name in enumerate(labels)}

In [ ]:
# ---------------------------------------------------------------------
# Epoch

# ---------------------------------------------------------------------

def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer,
    scaler,
    device: torch.device,
    cfg: Config,
    train: bool,
):
    model.train(train)
    total_loss = 0.0
    total_n = 0
    y_true: List[int] = []
    y_pred: List[int] = []

    if train:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        images = batch["image"].to(device, non_blocking=True)
        metadata = batch["metadata"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(cfg.amp and device.type == "cuda"),
        ):
            out = model(images, metadata, cfg)
            loss = F.cross_entropy(out["logits"], labels)
            loss_for_backward = loss / cfg.grad_accum

        if train:
            scaler.scale(loss_for_backward).backward()
            if (step + 1) % cfg.grad_accum == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                if cfg.max_grad_norm > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        bs = labels.shape[0]
        total_loss += float(loss.detach().item()) * bs
        total_n += bs
        pred = out["logits"].argmax(dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())

    metrics = compute_metrics(np.asarray(y_true), np.asarray(y_pred))
    metrics["loss"] = total_loss / max(total_n, 1)
    return metrics, np.asarray(y_true), np.asarray(y_pred)

In [ ]:
# ---------------------------------------------------------------------
# Intermediate check

# ---------------------------------------------------------------------

@torch.no_grad()
def run_intermediate_check(model, loader, device, cfg, output_dir):
    model.eval()
    batch = next(iter(loader))
    images = batch["image"].to(device)
    metadata = batch["metadata"].to(device)
    out = model(images, metadata, cfg, return_attention=True)

    print("\n" + "=" * 90)
    print("INTERMEDIATE MODEL RESULTS")
    print("=" * 90)
    print(f"Input clinical image tensor       : {tuple(images.shape)}")
    print(f"Global resized input              : ({images.shape[0]}, 3, {cfg.global_size}, {cfg.global_size})")
    print(f"Global Swin feature map           : {tuple(out['global_feature_map'].shape)}")
    print(f"Global tokens                     : {tuple(out['global_tokens'].shape)}")
    print(f"LSM lesion crops                  : {tuple(out['lesion_crops'].shape)}")
    print(f"LSM lesion boxes                  : {tuple(out['lesion_boxes'].shape)}")
    print(f"Local lesion tokens               : {tuple(out['local_tokens'].shape)}")
    print(f"Clinical global feature gC        : {tuple(out['gC'].shape)}")
    print(f"Metadata input vector             : {tuple(metadata.shape)}")
    print(f"Metadata feature gM               : {tuple(out['gM'].shape)}")
    print(f"Fused clinical+metadata feature   : {tuple(out['fused'].shape)}")
    print(f"Classifier logits                 : {tuple(out['logits'].shape)}")
    print(f"LSM score range                   : {out['lesion_scores'].min().item():.6f} .. {out['lesion_scores'].max().item():.6f}")
    print(f"CSF global→local attention       : {tuple(out['csf_global_to_local_attention'].shape)}")
    print(f"CSF local→global attention       : {tuple(out['csf_local_to_global_attention'].shape)}")
    print(f"CMF clinical↔metadata attention  : {tuple(out['cmf_attention'].shape)}")
    print("=" * 90 + "\n")

    save_lsm_preview(
        images[0].cpu(),
        out["lesion_boxes"][out["lesion_boxes"][:, 0] == 0],
        output_dir / "lsm_lesion_proposals.png",
    )

In [ ]:
def train_model(frame, cfg, label_to_idx, metadata_encoder, output_root):
    device = get_device()
    run_dir = output_root / "training_run"
    run_dir.mkdir(parents=True, exist_ok=True)

    train_idx, val_idx = make_split(frame, cfg.val_fraction, cfg.seed)
    train_df = frame.iloc[train_idx].reset_index(drop=True)
    val_df = frame.iloc[val_idx].reset_index(drop=True)

    metadata_encoder.fit(train_df)
    train_meta = metadata_encoder.transform(train_df)
    val_meta = metadata_encoder.transform(val_df)
    metadata_encoder.save(run_dir / "metadata_encoder.json")

    train_ds = PADUFES20Dataset(
        train_df,
        label_to_idx,
        build_train_transform(cfg.image_size),
        train_meta,
    )
    val_ds = PADUFES20Dataset(
        val_df,
        label_to_idx,
        build_eval_transform(cfg.image_size),
        val_meta,
    )

    print(f"\n[SPLIT] train={len(train_ds)}, val={len(val_ds)}")
    print(f"[META] one-hot input dimension={metadata_encoder.output_dim}")
    print(f"[META] encoded columns={len(metadata_encoder.columns)}")
    print("\n[SPLIT CLASS DISTRIBUTION]")
    class_names = [x for x, _ in sorted(label_to_idx.items(), key=lambda z: z[1])]
    for cname in class_names:
        train_count = int((train_df["_label_str"] == cname).sum())
        val_count = int((val_df["_label_str"] == cname).sum())
        print(f"  {cname:20s} | train={train_count:4d} | val={val_count:4d}")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(cfg.num_workers > 0),
        drop_last=(cfg.batch_size > 1),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(cfg.num_workers > 0),
        drop_last=False,
    )

    save_training_preview(train_ds, run_dir / "augmentation_preview.png")

    model = RemixPADClinicalMetadata(
        num_classes=len(label_to_idx),
        metadata_input_dim=metadata_encoder.output_dim,
        cfg=cfg,
    ).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.lr,
        momentum=0.9,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.epochs,
        eta_min=0.0,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))

    run_intermediate_check(model, val_loader, device, cfg, run_dir)
    save_single_image_intermediates(model, val_ds, val_df, label_to_idx, device, cfg, run_dir, image_index=0, stage="before_training")

    history = []
    best_acc = -1.0
    best_f1 = -1.0
    best_epoch = -1

    for epoch in range(1, cfg.epochs + 1):
        lr_now = optimizer.param_groups[0]["lr"]

        train_metrics, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, device, cfg, train=True
        )
        with torch.no_grad():
            val_metrics, y_true, y_pred = run_epoch(
                model, val_loader, optimizer, scaler, device, cfg, train=False
            )
        scheduler.step()

        row = {
            "epoch": epoch, "lr": lr_now,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_macro_precision": train_metrics["macro_precision"],
            "train_macro_recall": train_metrics["macro_recall"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
        }
        history.append(row)

        print(
            f"\nEpoch {epoch:02d}/{cfg.epochs} | LR={lr_now:.7f}\n"
            f"  TRAIN | Loss={train_metrics['loss']:.5f} | Acc={train_metrics['accuracy']*100:.2f}% | "
            f"BACC={train_metrics['balanced_accuracy']:.4f} | F1={train_metrics['macro_f1']:.4f} | "
            f"Precision={train_metrics['macro_precision']:.4f} | Recall={train_metrics['macro_recall']:.4f}\n"
            f"  VAL   | Loss={val_metrics['loss']:.5f} | Acc={val_metrics['accuracy']*100:.2f}% | "
            f"BACC={val_metrics['balanced_accuracy']:.4f} | F1={val_metrics['macro_f1']:.4f} | "
            f"Precision={val_metrics['macro_precision']:.4f} | Recall={val_metrics['macro_recall']:.4f}"
        )

        # Save the latest complete checkpoint every epoch for recovery/resume.
        latest_payload = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epoch": epoch,
            "label_to_idx": label_to_idx,
            "config": asdict(cfg),
            "paper_reference": PAPER,
            "latest_val_accuracy": val_metrics["accuracy"],
            "latest_val_macro_f1": val_metrics["macro_f1"],
            "latest_val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "latest_val_macro_precision": val_metrics["macro_precision"],
            "latest_val_macro_recall": val_metrics["macro_recall"],
            "metadata_columns": metadata_encoder.columns,
            "checkpoint_type": "last",
        }
        torch.save(latest_payload, run_dir / "last.pt")

        # Save periodic complete checkpoints every 10 epochs and at the final epoch.
        if (epoch % 10 == 0) or (epoch == cfg.epochs):
            periodic_dir = run_dir / "checkpoints"
            periodic_dir.mkdir(parents=True, exist_ok=True)
            periodic_payload = dict(latest_payload)
            periodic_payload["checkpoint_type"] = "periodic"
            periodic_path = periodic_dir / f"checkpoint_epoch_{epoch:03d}.pt"
            torch.save(periodic_payload, periodic_path)
            print(f"  SAVED CHECKPOINT: {periodic_path}")

        # Save/update best checkpoints when validation macro-F1 improves.
        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_acc = val_metrics["accuracy"]
            best_epoch = epoch
            class_names = [x for x, _ in sorted(label_to_idx.items(), key=lambda z: z[1])]

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "epoch": epoch,
                    "label_to_idx": label_to_idx,
                    "config": asdict(cfg),
                    "paper_reference": PAPER,
                    "best_val_accuracy": val_metrics["accuracy"],
                    "best_val_macro_f1": best_f1,
                    "best_val_balanced_accuracy": val_metrics["balanced_accuracy"],
                    "best_val_macro_precision": val_metrics["macro_precision"],
                    "best_val_macro_recall": val_metrics["macro_recall"],
                    "metadata_columns": metadata_encoder.columns,
                    "checkpoint_selection": "validation_macro_f1",
                },
                run_dir / "best.pt",
            )

            torch.save(
                {
                    "clinical_branch_state_dict": model.clinical.state_dict(),
                    "metadata_branch_state_dict": model.metadata.state_dict(),
                    "clinical_metadata_fusion_state_dict": model.cmf.state_dict(),
                    "epoch": epoch,
                    "best_val_macro_f1": best_f1,
                    "best_val_accuracy": val_metrics["accuracy"],
                    "best_val_balanced_accuracy": val_metrics["balanced_accuracy"],
                    "best_val_macro_precision": val_metrics["macro_precision"],
                    "best_val_macro_recall": val_metrics["macro_recall"],
                    "label_to_idx": label_to_idx,
                    "metadata_input_dim": metadata_encoder.output_dim,
                    "metadata_columns": metadata_encoder.columns,
                    "imagenet_pretrained": cfg.swin_pretrained,
                    "source_dataset": "PAD-UFES-20",
                    "target_use": "Derm7pt full RemixFormer++ initialization",
                    "checkpoint_selection": "validation_macro_f1",
                },
                run_dir / "best_transfer_encoder.pt",
            )

            save_confusion_matrix(
                y_true, y_pred, class_names,
                run_dir / "best_confusion_matrix.png",
                f"Best Validation Macro-F1 Confusion Matrix | Epoch {epoch}",
            )
            print(f"  NEW BEST: Val Macro-F1={best_f1:.6f} at epoch {best_epoch}")
            print_detailed_results(y_true, y_pred, class_names, prefix=f"BEST VALIDATION @ EPOCH {epoch}")

    # Reload the best full model and generate the same single-image intermediate visuals
    # after training, so the visual comparison is before-vs-best on exactly one image.
    best_ckpt_path = run_dir / "best.pt"
    if best_ckpt_path.exists():
        try:
            best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        except TypeError:
            best_ckpt = torch.load(best_ckpt_path, map_location=device)
        model.load_state_dict(best_ckpt["model_state_dict"], strict=True)
        save_single_image_intermediates(model, val_ds, val_df, label_to_idx, device, cfg, run_dir, image_index=0, stage="best_model")

    # Final detailed report from the saved BEST model.
    if best_ckpt_path.exists():
        with torch.no_grad():
            best_val_metrics, best_y_true, best_y_pred = run_epoch(
                model, val_loader, optimizer, scaler, device, cfg, train=False
            )
        best_class_names = [x for x, _ in sorted(label_to_idx.items(), key=lambda z: z[1])]
        final_cm = print_detailed_results(best_y_true, best_y_pred, best_class_names, prefix="FINAL BEST MODEL VALIDATION")
        save_confusion_matrix(
            best_y_true, best_y_pred, best_class_names,
            run_dir / "best_confusion_matrix.png",
            f"Best Validation Macro-F1 Confusion Matrix | Epoch {best_epoch}",
        )

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(run_dir / "training_history.csv", index=False)

    fig = plt.figure(figsize=(12, 5))
    ax = fig.add_subplot(111)
    ax.plot(hist_df["epoch"], hist_df["train_loss"], label="train loss")
    ax.plot(hist_df["epoch"], hist_df["val_loss"], label="val loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("RemixFormer++ PAD-UFES-20 clinical + metadata loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "loss_curve.png", dpi=180)
    plt.close(fig)

    fig = plt.figure(figsize=(12, 5))
    ax = fig.add_subplot(111)
    ax.plot(hist_df["epoch"], hist_df["train_accuracy"] * 100, label="train accuracy")
    ax.plot(hist_df["epoch"], hist_df["val_accuracy"] * 100, label="val accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("RemixFormer++ PAD-UFES-20 clinical + metadata accuracy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "accuracy_curve.png", dpi=180)
    plt.close(fig)

    # Additional metric curves. Epoch ticks are forced to whole numbers.
    metric_specs = [
        ("balanced_accuracy", "Balanced Accuracy", "balanced_accuracy_curve.png"),
        ("macro_f1", "Macro F1", "macro_f1_curve.png"),
        ("macro_precision", "Macro Precision", "macro_precision_curve.png"),
        ("macro_recall", "Macro Recall", "macro_recall_curve.png"),
    ]
    from matplotlib.ticker import MaxNLocator
    for metric, title, filename in metric_specs:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(hist_df["epoch"], hist_df[f"train_{metric}"], marker="o", label=f"train {title.lower()}")
        ax.plot(hist_df["epoch"], hist_df[f"val_{metric}"], marker="o", label=f"val {title.lower()}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.set_title(f"RemixFormer++ PAD-UFES-20 {title}")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.legend()
        fig.tight_layout()
        fig.savefig(run_dir / filename, dpi=180)
        plt.close(fig)

    detailed_summary = {
        "best_epoch": int(best_epoch),
        "best_validation_accuracy": float(best_acc),
        "best_validation_macro_f1": float(best_f1),
        "best_validation_metrics": {k: float(v) for k, v in best_val_metrics.items()},
        "confusion_matrix": final_cm.tolist(),
        "class_names": best_class_names,
        "checkpoint_policy": "best + last + periodic",
    }
    (run_dir / "best_model_detailed_results.json").write_text(
        json.dumps(detailed_summary, indent=2, default=str), encoding="utf-8"
    )

    summary = {
        "dataset": "PAD-UFES-20",
        "modality": "clinical + metadata",
        "split": "single_stratified_holdout",
        "val_fraction": cfg.val_fraction,
        "best_val_accuracy": float(best_acc),
        "best_val_macro_f1": float(best_f1),
        "best_macro_f1_epoch": int(best_epoch),
        "best_full_checkpoint": "best.pt",
        "best_transfer_checkpoint": "best_transfer_encoder.pt",
        "checkpoint_policy": "best + last + periodic, best selected by validation macro-F1",
        "epochs": cfg.epochs,
        "num_classes": len(label_to_idx),
        "label_to_idx": label_to_idx,
        "metadata_input_dim": metadata_encoder.output_dim,
        "metadata_columns": metadata_encoder.columns,
        "device": str(device),
        "paper_reference": PAPER,
        "active_config": asdict(cfg),
    }
    (run_dir / "summary.json").write_text(
        json.dumps(summary, indent=2, default=str),
        encoding="utf-8",
    )
    return summary

## 2. REAL 50-EPOCH KAGGLE RUN

This notebook is configured for **50 epochs**. It prints TRAIN and VAL metrics for every epoch, saves the best model according to validation Macro-F1, continuously overwrites `last.pt`, and saves complete periodic checkpoints every 10 epochs plus the final epoch.

The CMF attention figure writes the numeric attention value inside every Clinical/Metadata cell. The confusion matrix writes the integer count inside every cell.


In [ ]:
# =========================
# DATA PREPARATION + TRAINING
# =========================

frame, csv_path, image_col, label_col = prepare_dataframe(
    root,
    metadata_csv=cfg.metadata_csv,
    explicit_image_column=None,
    explicit_label_column=None,
    max_samples=cfg.max_samples,
)

# Image audit
modes = {}
sizes = {}
for p in frame["_resolved_image"].head(100).tolist():
    try:
        with Image.open(p) as im:
            modes[im.mode] = modes.get(im.mode, 0) + 1
            sizes[str(im.size)] = sizes.get(str(im.size), 0) + 1
    except Exception:
        pass
print("\n[IMAGE AUDIT] Modes:", modes)
print("[IMAGE AUDIT] Example resolutions:", dict(list(sizes.items())[:15]))

label_to_idx = build_label_map(frame)
print("\n[LABEL MAP]")
for k, v in label_to_idx.items():
    print(f"  {v}: {k}")

frame[["_resolved_image", "_label_str"]].to_csv(
    output / "resolved_manifest.csv", index=False
)

excluded = [image_col, label_col, "_resolved_image", "_label_str"]
metadata_encoder = MetadataEncoder(excluded=excluded)

(output / "config.json").write_text(
    json.dumps({
        "paper_reference": PAPER,
        "active_config": asdict(cfg),
        "metadata_csv": str(csv_path),
        "image_column": image_col,
        "label_column": label_col,
    }, indent=2, default=str),
    encoding="utf-8",
)

summary = train_model(
    frame=frame,
    cfg=cfg,
    label_to_idx=label_to_idx,
    metadata_encoder=metadata_encoder,
    output_root=output,
)

print("\nTraining complete.")
print(json.dumps(summary, indent=2, default=str))
print(f"Outputs saved to: {output}")


[DATA] metadata CSV       : /kaggle/input/datasets/abtahimajeed/pad-ufes-20/metadata.csv
[DATA] CSV rows            : 2298
[DATA] Usable rows         : 2298
[DATA] Missing images      : 0
[DATA] Image column        : img_id
[DATA] Label column        : diagnostic
[DATA] Class distribution:
_label_str
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64
[DATA] Image files indexed : 4596

[IMAGE AUDIT] Modes: {'RGB': 36, 'RGBA': 64}
[IMAGE AUDIT] Example resolutions: {'(1089, 1089)': 1, '(1325, 1325)': 1, '(640, 640)': 3, '(2118, 2117)': 1, '(896, 896)': 1, '(499, 499)': 1, '(514, 514)': 1, '(1726, 1726)': 1, '(1567, 1567)': 1, '(2022, 2022)': 1, '(1411, 1411)': 1, '(751, 751)': 1, '(880, 880)': 1, '(408, 408)': 1, '(1806, 1806)': 1}

[LABEL MAP]
  0: ACK
  1: BCC
  2: MEL
  3: NEV
  4: SCC
  5: SEK

[SPLIT] train=1838, val=460
[META] one-hot input dimension=254
[META] encoded columns=22

[SPLIT CLASS DISTRIBUTION]
  ACK                  | train= 58

100%|██████████| 108M/108M [00:00<00:00, 181MB/s]
/tmp/ipykernel_22/643169944.py:76: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device.type == "cuda"))



INTERMEDIATE MODEL RESULTS
Input clinical image tensor       : (2, 3, 896, 896)
Global resized input              : (2, 3, 384, 384)
Global Swin feature map           : (2, 12, 12, 768)
Global tokens                     : (2, 144, 768)
LSM lesion crops                  : (8, 3, 224, 224)
LSM lesion boxes                  : (8, 5)
Local lesion tokens               : (2, 4, 768)
Clinical global feature gC        : (2, 768)
Metadata input vector             : (2, 254)
Metadata feature gM               : (2, 768)
Fused clinical+metadata feature   : (2, 768)
Classifier logits                 : (2, 6)
LSM score range                   : -0.280013 .. 0.391807
CSF global→local attention       : (2, 144, 4)
CSF local→global attention       : (2, 4, 144)
CMF clinical↔metadata attention  : (2, 2, 2)

[INTERMEDIATE] Saved single-image visualizations to: /kaggle/working/remix_padufes20_clinical_metadata_50ep_final/training_run/single_image_intermediate/before_training
[INTERMEDIATE] Image: /kaggle

In [ ]:
# ---------------------------------------------------------------------
# Verify BEST checkpoints after training
# ---------------------------------------------------------------------

run_dir = Path(OUTPUT_DIR) / "training_run"
best_full = run_dir / "best.pt"
best_transfer = run_dir / "best_transfer_encoder.pt"
last_checkpoint = run_dir / "last.pt"

print("\nCHECKPOINT VERIFICATION")
print("=" * 95)
print(f"Best full model checkpoint       : {'FOUND' if best_full.exists() else 'MISSING'}")
print(f"Best transfer encoder checkpoint: {'FOUND' if best_transfer.exists() else 'MISSING'}")
print(f"Last checkpoint                  : {'FOUND' if last_checkpoint.exists() else 'MISSING'}")
periodic_files = sorted((run_dir / "checkpoints").glob("checkpoint_epoch_*.pt")) if (run_dir / "checkpoints").exists() else []
print(f"Periodic checkpoints             : {len(periodic_files)} found")

if not best_full.exists() or not best_transfer.exists() or not last_checkpoint.exists():
    raise RuntimeError("Best checkpoint files are missing.")

try:
    ckpt = torch.load(best_transfer, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(best_transfer, map_location="cpu")

print("Best transfer epoch            :", ckpt["epoch"])
print("Best transfer Macro-F1         :", f"{ckpt['best_val_macro_f1']:.6f}")
print("Best transfer accuracy         :", f"{ckpt['best_val_accuracy']*100:.2f}%")
print("Best transfer BACC             :", f"{ckpt['best_val_balanced_accuracy']:.6f}")
print("Best transfer precision        :", f"{ckpt['best_val_macro_precision']:.6f}")
print("Best transfer recall           :", f"{ckpt['best_val_macro_recall']:.6f}")
print("ImageNet initialization        :", ckpt["imagenet_pretrained"])
print("Verification                   : PASSED")
print("=" * 95)



CHECKPOINT VERIFICATION
Best full model checkpoint       : FOUND
Best transfer encoder checkpoint: FOUND
Last checkpoint                  : FOUND
Periodic checkpoints             : 5 found
Best transfer epoch            : 48
Best transfer Macro-F1         : 0.421703
Best transfer accuracy         : 69.57%
Best transfer BACC             : 0.429543
Best transfer precision        : 0.462449
Best transfer recall           : 0.429543
ImageNet initialization        : True
Verification                   : PASSED


In [ ]:
# ---------------------------------------------------------------------
# Final result inventory and complete printed summary
# ---------------------------------------------------------------------

run_dir = Path(OUTPUT_DIR) / "training_run"
print("\nGENERATED RESULTS")
print("=" * 95)
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(run_dir))

print("\nCHECKPOINT POLICY")
print("  best.pt                    = full model + optimizer + scheduler, selected by validation macro-F1")
print("  best_transfer_encoder.pt  = clinical + metadata + CMF weights for later Derm7pt transfer")
print("  last.pt                    = latest complete model + optimizer + scheduler, overwritten every epoch")
print("  checkpoints/checkpoint_epoch_XXX.pt = complete periodic checkpoints saved every 10 epochs and at the final epoch")
print("  Confusion matrix cells contain the actual integer counts.")
print("  CMF attention cells contain the actual attention values.")
print("  All epoch metrics and final best-model metrics are printed above and saved in training_history.csv / summary.json / best_model_detailed_results.json.")
print("  Saved plots include loss, accuracy, balanced accuracy, macro-F1, macro-precision, macro-recall, best confusion matrix, and all single-image intermediate visualizations.")



GENERATED RESULTS
accuracy_curve.png
augmentation_preview.png
balanced_accuracy_curve.png
best.pt
best_confusion_matrix.png
best_model_detailed_results.json
best_transfer_encoder.pt
checkpoints/checkpoint_epoch_010.pt
checkpoints/checkpoint_epoch_020.pt
checkpoints/checkpoint_epoch_030.pt
checkpoints/checkpoint_epoch_040.pt
checkpoints/checkpoint_epoch_050.pt
last.pt
loss_curve.png
lsm_lesion_proposals.png
macro_f1_curve.png
macro_precision_curve.png
macro_recall_curve.png
metadata_encoder.json
single_image_intermediate/before_training/01_original.png
single_image_intermediate/before_training/02_lsm_attention_score_heatmap.png
single_image_intermediate/before_training/03_attention_sampling_boxes.png
single_image_intermediate/before_training/04_lsm_pooled_score.png
single_image_intermediate/before_training/05_selected_lesion_crops.png
single_image_intermediate/before_training/06_global_feature_activation.png
single_image_intermediate/before_training/07_local_feature_activation.png
sing

## 3. SAVED OUTPUTS

The run produces the complete epoch history, all requested plots, detailed best-model metrics, numeric attention visualizations, and:

- `best.pt`
- `best_transfer_encoder.pt`
- `last.pt`
- `checkpoints/checkpoint_epoch_010.pt` through the final 10-epoch interval and final epoch

The best checkpoint is selected using validation Macro-F1.
